<a href="https://colab.research.google.com/github/dorhoffman/SWINGPULSE/blob/main/notebooks/03_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np

PROJECT_PATH = "/content/drive/MyDrive/SWINGPULSE"

CLEAN_PATH = (
    f"{PROJECT_PATH}/data/clean/"
    "SWINGPULSE_CLEAN_DATASET.csv"
)

FEATURES_FOLDER = f"{PROJECT_PATH}/data/features"

FEATURES_PATH = (
    f"{FEATURES_FOLDER}/"
    "SWINGPULSE_FEATURES_DATASET.csv"
)

os.makedirs(FEATURES_FOLDER, exist_ok=True)

print("Clean file exists:", os.path.exists(CLEAN_PATH))
print("Features output:", FEATURES_PATH)

Mounted at /content/drive
Clean file exists: True
Features output: /content/drive/MyDrive/SWINGPULSE/data/features/SWINGPULSE_FEATURES_DATASET.csv


In [2]:
def calculate_rsi(series, period=14):
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period
    ).mean()

    avg_loss = loss.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period
    ).mean()

    rs = avg_gain / avg_loss

    return 100 - (100 / (1 + rs))


def add_technical_features(stock_df):
    stock_df = stock_df.sort_values("Date").copy()

    # תשואות ושינויי נפח
    stock_df["Daily_Return"] = stock_df["Close"].pct_change()
    stock_df["Volume_Change"] = stock_df["Volume"].pct_change()

    # SMA
    stock_df["SMA_20"] = stock_df["Close"].rolling(
        window=20
    ).mean()

    stock_df["SMA_50"] = stock_df["Close"].rolling(
        window=50
    ).mean()

    stock_df["SMA_200"] = stock_df["Close"].rolling(
        window=200
    ).mean()

    # EMA
    stock_df["EMA_20"] = stock_df["Close"].ewm(
        span=20,
        adjust=False
    ).mean()

    stock_df["EMA_50"] = stock_df["Close"].ewm(
        span=50,
        adjust=False
    ).mean()

    stock_df["EMA_200"] = stock_df["Close"].ewm(
        span=200,
        adjust=False
    ).mean()

    # RSI
    stock_df["RSI_14"] = calculate_rsi(
        stock_df["Close"],
        period=14
    )

    # MACD
    ema_12 = stock_df["Close"].ewm(
        span=12,
        adjust=False
    ).mean()

    ema_26 = stock_df["Close"].ewm(
        span=26,
        adjust=False
    ).mean()

    stock_df["MACD"] = ema_12 - ema_26

    stock_df["MACD_Signal"] = stock_df["MACD"].ewm(
        span=9,
        adjust=False
    ).mean()

    stock_df["MACD_Histogram"] = (
        stock_df["MACD"]
        - stock_df["MACD_Signal"]
    )

    # Bollinger Bands
    rolling_mean = stock_df["Close"].rolling(
        window=20
    ).mean()

    rolling_std = stock_df["Close"].rolling(
        window=20
    ).std()

    stock_df["BB_Middle"] = rolling_mean
    stock_df["BB_Upper"] = rolling_mean + 2 * rolling_std
    stock_df["BB_Lower"] = rolling_mean - 2 * rolling_std

    stock_df["BB_Width"] = (
        stock_df["BB_Upper"]
        - stock_df["BB_Lower"]
    ) / stock_df["BB_Middle"]

    # ATR
    previous_close = stock_df["Close"].shift(1)

    true_range = pd.concat(
        [
            stock_df["High"] - stock_df["Low"],
            (stock_df["High"] - previous_close).abs(),
            (stock_df["Low"] - previous_close).abs()
        ],
        axis=1
    ).max(axis=1)

    stock_df["ATR_14"] = true_range.rolling(
        window=14
    ).mean()

    # תנודתיות
    stock_df["Volatility_20"] = (
        stock_df["Daily_Return"]
        .rolling(window=20)
        .std()
    )

    # Features יחסיים
    stock_df["Price_to_SMA20"] = (
        stock_df["Close"] / stock_df["SMA_20"]
    )

    stock_df["Price_to_SMA50"] = (
        stock_df["Close"] / stock_df["SMA_50"]
    )

    stock_df["Price_to_EMA20"] = (
        stock_df["Close"] / stock_df["EMA_20"]
    )

    stock_df["EMA20_to_EMA50"] = (
        stock_df["EMA_20"] / stock_df["EMA_50"]
    )

    return stock_df

In [3]:
aapl_parts = []

for chunk in pd.read_csv(
    CLEAN_PATH,
    chunksize=100_000,
    parse_dates=["Date"],
    low_memory=False
):
    filtered = chunk[
        chunk["Symbol"] == "AAPL"
    ]

    if not filtered.empty:
        aapl_parts.append(filtered)

aapl_data = pd.concat(
    aapl_parts,
    ignore_index=True
)

aapl_features = add_technical_features(
    aapl_data
)

display(
    aapl_features[
        [
            "Date",
            "Symbol",
            "Close",
            "SMA_20",
            "EMA_20",
            "RSI_14",
            "MACD",
            "MACD_Signal",
            "ATR_14",
            "Volatility_20"
        ]
    ].tail(20)
)

,Date,Symbol,Close,SMA_20,EMA_20,RSI_14,MACD,MACD_Signal,ATR_14,Volatility_20
10763,2023-08-24 04:00:00,AAPL,176.380005,181.812516,180.685073,39.218860,-3.181068,-3.030875,3.233502,0.015532
10764,2023-08-25 04:00:00,AAPL,178.610001,180.964720,180.487447,44.082997,-2.943953,-3.013491,3.059057,0.015481
10765,2023-08-28 04:00:00,AAPL,180.190002,180.164967,180.459119,47.300937,-2.598590,-2.930510,3.012887,0.015678
10766,2023-08-29 04:00:00,AAPL,184.119995,179.603656,180.807774,54.339636,-1.984888,-2.741386,3.118979,0.016716
10767,2023-08-30 04:00:00,AAPL,187.649994,179.370141,181.459414,59.563912,-1.199853,-2.433079,3.160712,0.017125
10768,2023-08-31 04:00:00,AAPL,187.869995,179.218031,182.069945,59.872066,-0.553574,-2.057178,3.129998,0.017067
10769,2023-09-01 04:00:00,AAPL,189.460007,179.603802,182.773761,62.118955,0.085918,-1.628559,3.106427,0.013010
10770,2023-09-05 04:00:00,AAPL,189.699997,180.158361,183.433402,62.460619,0.605109,-1.181825,3.102141,0.012188
10771,2023-09-06 04:00:00,AAPL,182.910004,180.325985,183.383555,48.995884,0.463335,-0.852793,3.544284,0.014947
10772,2023-09-07 04:00:00,AAPL,177.559998,180.306499,182.828930,41.419440,-0.079802,-0.698195,3.925714,0.016289


In [4]:
feature_columns = [
    "Daily_Return",
    "Volume_Change",
    "SMA_20",
    "SMA_50",
    "SMA_200",
    "EMA_20",
    "EMA_50",
    "EMA_200",
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "MACD_Histogram",
    "BB_Middle",
    "BB_Upper",
    "BB_Lower",
    "BB_Width",
    "ATR_14",
    "Volatility_20",
    "Price_to_SMA20",
    "Price_to_SMA50",
    "Price_to_EMA20",
    "EMA20_to_EMA50"
]

print(
    aapl_features[
        feature_columns
    ].isnull().sum()
)

Daily_Return        1
Volume_Change       1
SMA_20             19
SMA_50             49
SMA_200           199
EMA_20              0
EMA_50              0
EMA_200             0
RSI_14             14
MACD                0
MACD_Signal         0
MACD_Histogram      0
BB_Middle          19
BB_Upper           19
BB_Lower           19
BB_Width           19
ATR_14             13
Volatility_20      20
Price_to_SMA20     19
Price_to_SMA50     49
Price_to_EMA20      0
EMA20_to_EMA50      0
dtype: int64


In [5]:
if os.path.exists(FEATURES_PATH):
    os.remove(FEATURES_PATH)

# קודם נאסוף רשימת סימולים
symbols = set()

for chunk in pd.read_csv(
    CLEAN_PATH,
    usecols=["Symbol"],
    chunksize=100_000
):
    symbols.update(
        chunk["Symbol"]
        .dropna()
        .unique()
    )

symbols = sorted(symbols)

print("Number of stocks:", len(symbols))

Number of stocks: 503


In [6]:
first_write = True
processed_stocks = 0
failed_stocks = []
total_feature_rows = 0

for index, symbol in enumerate(symbols, start=1):

    try:
        symbol_parts = []

        for chunk in pd.read_csv(
            CLEAN_PATH,
            chunksize=100_000,
            parse_dates=["Date"],
            low_memory=False
        ):
            filtered = chunk[
                chunk["Symbol"] == symbol
            ]

            if not filtered.empty:
                symbol_parts.append(filtered)

        if not symbol_parts:
            raise ValueError("No rows found")

        symbol_df = pd.concat(
            symbol_parts,
            ignore_index=True
        )

        symbol_features = add_technical_features(
            symbol_df
        )

        symbol_features.to_csv(
            FEATURES_PATH,
            mode="w" if first_write else "a",
            header=first_write,
            index=False
        )

        first_write = False
        processed_stocks += 1
        total_feature_rows += len(symbol_features)

        if index % 10 == 0:
            print(
                f"Processed {index}/{len(symbols)} stocks | "
                f"Rows written: {total_feature_rows:,}"
            )

        del symbol_df
        del symbol_features
        del symbol_parts

    except Exception as error:
        failed_stocks.append(
            {
                "Symbol": symbol,
                "Error": str(error)
            }
        )

print("Feature engineering completed")
print("Processed stocks:", processed_stocks)
print("Failed stocks:", len(failed_stocks))
print("Rows written:", total_feature_rows)

Processed 10/503 stocks | Rows written: 66,516
Processed 20/503 stocks | Rows written: 160,150
Processed 30/503 stocks | Rows written: 232,167
Processed 40/503 stocks | Rows written: 309,525
Processed 50/503 stocks | Rows written: 374,938
Processed 60/503 stocks | Rows written: 479,529
Processed 70/503 stocks | Rows written: 564,291
Processed 80/503 stocks | Rows written: 637,687
Processed 90/503 stocks | Rows written: 701,042
Processed 100/503 stocks | Rows written: 768,530
Processed 110/503 stocks | Rows written: 851,084
Processed 120/503 stocks | Rows written: 928,119
Processed 130/503 stocks | Rows written: 1,000,724
Processed 140/503 stocks | Rows written: 1,082,016
Processed 150/503 stocks | Rows written: 1,160,649
Processed 160/503 stocks | Rows written: 1,242,302
Processed 170/503 stocks | Rows written: 1,316,461
Processed 180/503 stocks | Rows written: 1,400,887
Processed 190/503 stocks | Rows written: 1,472,546
Processed 200/503 stocks | Rows written: 1,529,816
Processed 210/

In [7]:
features_sample = pd.read_csv(
    FEATURES_PATH,
    nrows=20,
    parse_dates=["Date"]
)

display(features_sample)

print("Columns:")
print(features_sample.columns.tolist())

print("\nFile size:")
print(
    f"{os.path.getsize(FEATURES_PATH) / 1024**2:.2f} MB"
)

,Date,Symbol,Open,High,Low,Close,Volume,Dividends,Stock_Splits,Daily_Return,...,BB_Middle,BB_Upper,BB_Lower,BB_Width,ATR_14,Volatility_20,Price_to_SMA20,Price_to_SMA50,Price_to_EMA20,EMA20_to_EMA50
0,1999-11-18 05:00:00,A,27.708214,30.448590,24.358870,26.794758,62546380,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000
1,1999-11-19 05:00:00,A,26.147735,26.185797,24.244699,24.587246,15234146,0.0,0.0,-0.082386,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.924871,0.995370
2,1999-11-22 05:00:00,A,25.158145,26.794758,24.396931,26.794758,6577870,0.0,0.0,0.089783,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.007150,0.995993
3,1999-11-23 05:00:00,A,25.881299,26.566393,24.358870,24.358870,5975611,0.0,0.0,-0.090909,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.923011,0.991410
4,1999-11-24 05:00:00,A,24.435000,25.538762,24.358877,25.005911,4843231,0.0,0.0,0.026563,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.952287,0.988806
5,1999-11-26 05:00:00,A,24.891718,25.272325,24.815597,25.082022,1729466,0.0,0.0,0.003044,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.959280,0.986734
6,1999-11-29 05:00:00,A,24.967845,25.843241,24.701418,25.652937,4074751,0.0,0.0,0.022762,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.982883,0.986193
7,1999-11-30 05:00:00,A,25.576813,26.147724,24.929781,25.690996,4310034,0.0,0.0,0.001484,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.985811,0.985853
8,1999-12-01 05:00:00,A,25.691002,26.452216,25.500698,26.147730,2957329,0.0,0.0,0.017778,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.003018,0.986586
9,1999-12-02 05:00:00,A,26.642514,27.403730,26.299967,26.870878,3069868,0.0,0.0,0.027656,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.027747,0.988820


Columns:
['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock_Splits', 'Daily_Return', 'Volume_Change', 'SMA_20', 'SMA_50', 'SMA_200', 'EMA_20', 'EMA_50', 'EMA_200', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'BB_Width', 'ATR_14', 'Volatility_20', 'Price_to_SMA20', 'Price_to_SMA50', 'Price_to_EMA20', 'EMA20_to_EMA50']

File size:
1963.68 MB


In [8]:
total_rows = 0
feature_missing = None

for chunk in pd.read_csv(
    FEATURES_PATH,
    chunksize=100_000,
    low_memory=False
):
    total_rows += len(chunk)

    current_missing = chunk[
        feature_columns
    ].isnull().sum()

    if feature_missing is None:
        feature_missing = current_missing
    else:
        feature_missing = feature_missing.add(
            current_missing,
            fill_value=0
        )

print(f"Total feature rows: {total_rows:,}")
print("\nMissing values in features:")
print(feature_missing.astype(int))

Total feature rows: 3,878,687

Missing values in features:
Daily_Return        503
Volume_Change      9553
SMA_20             9557
SMA_50            24647
SMA_200           99988
EMA_20                0
EMA_50                0
EMA_200               0
RSI_14             7050
MACD                  0
MACD_Signal           0
MACD_Histogram        0
BB_Middle          9557
BB_Upper           9557
BB_Lower           9557
BB_Width           9557
ATR_14             6539
Volatility_20     10060
Price_to_SMA20     9557
Price_to_SMA50    24647
Price_to_EMA20        0
EMA20_to_EMA50        0
dtype: int64
